In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [2]:
FIGURE_DIR = "../output/figures/regression"
TABLE_DIR = "../output/tables"
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

def save_figure(filename_stem, directory=FIGURE_DIR):
    """Save the current matplotlib figure as pdf, png and svg."""
    for ext in ["pdf", "png", "svg"]:
        plt.savefig(f"{directory}/{filename_stem}.{ext}", dpi=300, bbox_inches="tight")

def save_latex_table(models, model_names, filename_stem, directory=TABLE_DIR, label_map=None, **kwargs):
    """Combine a list of fitted statsmodels OLS results into one side-by-side LaTeX table."""
    table = summary_col(
        models,
        stars=True,
        model_names=model_names,
        info_dict={
            "N": lambda m: f"{int(m.nobs)}",
            "R2": lambda m: f"{m.rsquared:.3f}",
        },
        **kwargs
    )
    tex = table.as_latex()
    if label_map:
        for var, label in sorted(label_map.items(), key=lambda kv: len(kv[0]), reverse=True):
            tex = tex.replace(var, label)
            tex = tex.replace(var.replace('_', r'\_'), label)
    with open(f"{directory}/{filename_stem}.tex", "w") as f:
        f.write(tex)
    return table

In [3]:
df = pd.read_pickle("../output/data/merged_analysis_data_standardized.pkl")

In [4]:
industry_codes = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S"
]
industry_bool_cols = [f"industry_{c}" for c in industry_codes]

df["industry_fe"] = df[industry_bool_cols].idxmax(axis=1).str.replace("industry_", "")
df["industry_fe"] = pd.Categorical(df["industry_fe"])

# Build the five nested models

Each step adds one block of variables on top of the previous step, ending at the full
specification (identical to notebook 10a's model_full).

In [9]:
block_baseline = ["loc_work_gaussian_opening"]

block_city_cluster = ["distance_to_deak", "work_cluster_1", "work_cluster_0"]

block_company = ["productivity", "n_companies", "entropy"]

block_socioecon = ["arpu_high_ratio_open", "income_entr_open", "sex_fem_ratio_open"]

block_industry = ['C(industry_fe, Treatment("G"))']

formula_1 = "work_abs_diff ~ " + " + ".join(block_baseline)
formula_2 = formula_1 + " + " + " + ".join(block_city_cluster)
formula_3 = formula_2 + " + " + " + ".join(block_company)
formula_4 = formula_3 + " + " + " + ".join(block_socioecon)
formula_5 = formula_4 + " + " + " + ".join(block_industry)

formulas = [formula_1, formula_2, formula_3, formula_4, formula_5]
step_names = [
    "1. Baseline",
    "2. + City",
    "3. + Company",
    "4. + Socioecon.",
    "5. + Industry FE",
]

In [10]:
models = [smf.ols(formula=f, data=df).fit(cov_type="HC3") for f in formulas]

for name, model in zip(step_names, models):
    print(f"=== {name} ===")
    print(f"N = {int(model.nobs)}, R2 = {model.rsquared:.4f}, Adj. R2 = {model.rsquared_adj:.4f}\n")

=== 1. Baseline ===
N = 8861, R2 = 0.2652, Adj. R2 = 0.2651

=== 2. + City ===
N = 8861, R2 = 0.3341, Adj. R2 = 0.3338

=== 3. + Company ===
N = 8861, R2 = 0.3419, Adj. R2 = 0.3414

=== 4. + Socioecon. ===
N = 8861, R2 = 0.3495, Adj. R2 = 0.3487

=== 5. + Industry FE ===
N = 8861, R2 = 0.3539, Adj. R2 = 0.3519



# LaTeX regression table

In [15]:
formula_4

'work_abs_diff ~ loc_work_gaussian_opening + distance_to_deak + work_cluster_1 + work_cluster_0 + productivity + n_companies + entropy + arpu_high_ratio_open + income_entr_open + sex_fem_ratio_open'

In [13]:
industry_names = {
    'A': 'Agriculture, forestry and fishing',
    'B': 'Mining and quarrying',
    'C': 'Manufacturing',
    'D': 'Electricity, gas, steam and air conditioning supply',
    'E': 'Water supply; sewerage, waste management',
    'F': 'Construction',
    'G': 'Wholesale and retail trade',
    'H': 'Transportation and storage',
    'I': 'Accommodation and food service activities',
    'J': 'Information and communication',
    'K': 'Financial and insurance',
    'L': 'Real estate activities',
    'M': 'Professional, scientific and technical',
    'N': 'Administrative and support service',
    'O': 'Public administration and defence',
    'P': 'Education',
    'Q': 'Human health and social work',
    'R': 'Arts, entertainment and recreation',
    'S': 'Other service activities'
}

latex_label_map = {
    "loc_work_gaussian_opening": "Baseline work activity",
    "productivity": "Productivity",
    "n_companies": "Number of companies",
    "entropy":  "Industry entropy",
    "distance_to_deak": "Distance to centre",
    "work_cluster_1": "Day Shift cluster",
    "work_cluster_0": "Mixed Shift cluster",
    "arpu_high_ratio_open": "High income ratio",
    "income_entr_open": "Income entropy",
    "sex_fem_ratio_open": "Female ratio",
}

for letter, name in industry_names.items():
    latex_label_map[f'C(industry_fe, Treatment("G"))[T.{letter}]'] = name

In [14]:
save_latex_table(
    models,
    model_names=step_names,
    filename_stem="stepwise_regression_table",
    label_map=latex_label_map,
)

,1. Baseline,2. + City,3. + Company,4. + Socioecon.,5. + Industry FE
Intercept,0.0000,-0.0619***,-0.0545***,-0.0441***,-0.0066
,(0.0091),(0.0127),(0.0126),(0.0122),(0.0177)
loc_work_gaussian_opening,-0.5150***,-0.6167***,-0.6433***,-0.6095***,-0.6114***
,(0.0220),(0.0253),(0.0263),(0.0319),(0.0319)
work_cluster_1[T.True],,-0.3657***,-0.3705***,-0.3807***,-0.3806***
,,(0.0262),(0.0263),(0.0273),(0.0273)
work_cluster_0[T.True],,0.3575***,0.3381***,0.3125***,0.3106***
,,(0.0227),(0.0222),(0.0214),(0.0215)
distance_to_deak,,-0.0959***,-0.0878***,-0.1038***,-0.1094***
,,(0.0089),(0.0086),(0.0090),(0.0092)
